
# Part 2 — Netflix User-Activity Messaging System

Course: Database Management Tools in Python | Spring 2026



# Import Required Libraries


In [41]:

from collections import deque
from datetime import datetime, timedelta



# Base Classes


In [42]:

class SimpleTopic:

    def __init__(self, name):
        self.name = name
        self.messages = deque()
        self.offset = 0

    def publish(self, message):
        message['offset'] = self.offset
        self.messages.append(message)
        self.offset += 1

    def read_from(self, offset=0):
        return [msg for msg in self.messages if msg['offset'] >= offset]


In [43]:

class SimpleProducer:

    def __init__(self, topic):
        self.topic = topic


In [44]:

class SimpleConsumer:

    def __init__(self, topic):
        self.topic = topic



# AdvancedTopic Class


In [45]:

class AdvancedTopic(SimpleTopic):

    def __init__(self, name):
        super().__init__(name)
        self.message_count = 0

    def publish(self, message):
        message['offset'] = self.offset
        self.messages.append(message)
        self.offset += 1
        self.message_count += 1

    def view_all_messages(self):
        return list(self.messages)

    def get_messages_by_offset(self, offset):

        for msg in self.messages:
            if msg['offset'] == offset:
                return msg

        return None

    def delete_messages(self, offset):

        self.messages = deque([
            msg for msg in self.messages
            if msg['offset'] != offset
        ])

    def get_latest_message(self, user_id=None):

        if not self.messages:
            return None

        if user_id is None:
            return self.messages[-1]

        user_messages = [
            msg for msg in self.messages
            if msg['user_id'] == user_id
        ]

        if user_messages:
            return user_messages[-1]

        return None

    def get_message_count(self):
        return self.message_count

    def read_from(self, offset=None, user_id=None):

        result = []

        while self.messages:

            msg = self.messages.popleft()

            if offset is not None:

                if msg['offset'] >= offset:
                    result.append(msg)

            elif user_id is not None:

                if msg['user_id'] == user_id:
                    result.append(msg)

            else:
                result.append(msg)

        return result



# AdvancedProducer Class


In [46]:

class AdvancedProducer(SimpleProducer):

    def send_activity(self,
                      user_id,
                      title,
                      activity_status,
                      **metadata):

        message = {
            'ingested_at': datetime.now().isoformat(),
            'user_id': user_id,
            'title': title,
            'activity_status': activity_status
        }

        message.update(metadata)

        self.topic.publish(message)

        print('Message Published:', message)



# AdvancedConsumer Class


In [47]:

class AdvancedConsumer(SimpleConsumer):

    def poll_messages(self):

        messages = self.topic.read_from()

        print('\nConsumed Messages:')

        for msg in messages:
            print(msg)

        return messages

    def poll_by_user(self, user_id):

        messages = self.topic.read_from(user_id=user_id)

        print(f'\nMessages for User {user_id}:')

        for msg in messages:
            print(msg)

        return messages

    def view_latest_activity(self, user_id=None):

        latest = self.topic.get_latest_message(user_id)

        print('\nLatest Activity:')
        print(latest)

        return latest



# Create Topic, Producer, and Consumer


In [28]:

netflix_topic = AdvancedTopic('NetflixActivityTopic')

producer = AdvancedProducer(netflix_topic)

consumer = AdvancedConsumer(netflix_topic)



# Generate and Send Activity Messages


In [48]:

base_time = datetime.now()

activities = [
    ('U001', 'Stranger Things', 'watching', 'mobile'),
    ('U002', 'Money Heist', 'watching', 'tv'),
    ('U003', 'Wednesday', 'paused', 'tablet'),
    ('U001', 'Stranger Things', 'paused', 'mobile'),
    ('U004', 'Dark', 'watching', 'laptop'),
    ('U002', 'Money Heist', 'closed', 'tv'),
    ('U005', 'Narcos', 'watching', 'mobile'),
    ('U003', 'Wednesday', 'watching', 'tablet'),
    ('U001', 'Stranger Things', 'closed', 'mobile'),
    ('U006', 'The Crown', 'watching', 'smart_tv')
]

for i, activity in enumerate(activities):

    event_time = (
        base_time + timedelta(seconds=i*10)
    ).isoformat()

    producer.send_activity(
        user_id=activity[0],
        title=activity[1],
        activity_status=activity[2],
        device=activity[3],
        event_time=event_time
    )


Message Published: {'ingested_at': '2026-05-12T03:24:48.790582', 'user_id': 'U001', 'title': 'Stranger Things', 'activity_status': 'watching', 'device': 'mobile', 'event_time': '2026-05-12T03:24:48.790378', 'offset': 10}
Message Published: {'ingested_at': '2026-05-12T03:24:48.790785', 'user_id': 'U002', 'title': 'Money Heist', 'activity_status': 'watching', 'device': 'tv', 'event_time': '2026-05-12T03:24:58.790378', 'offset': 11}
Message Published: {'ingested_at': '2026-05-12T03:24:48.790819', 'user_id': 'U003', 'title': 'Wednesday', 'activity_status': 'paused', 'device': 'tablet', 'event_time': '2026-05-12T03:25:08.790378', 'offset': 12}
Message Published: {'ingested_at': '2026-05-12T03:24:48.790846', 'user_id': 'U001', 'title': 'Stranger Things', 'activity_status': 'paused', 'device': 'mobile', 'event_time': '2026-05-12T03:25:18.790378', 'offset': 13}
Message Published: {'ingested_at': '2026-05-12T03:24:48.790873', 'user_id': 'U004', 'title': 'Dark', 'activity_status': 'watching', 'd


# View All Messages


In [49]:

all_messages = netflix_topic.view_all_messages()

for msg in all_messages:
    print(msg)


{'ingested_at': '2026-05-12T03:24:48.790582', 'user_id': 'U001', 'title': 'Stranger Things', 'activity_status': 'watching', 'device': 'mobile', 'event_time': '2026-05-12T03:24:48.790378', 'offset': 10}
{'ingested_at': '2026-05-12T03:24:48.790785', 'user_id': 'U002', 'title': 'Money Heist', 'activity_status': 'watching', 'device': 'tv', 'event_time': '2026-05-12T03:24:58.790378', 'offset': 11}
{'ingested_at': '2026-05-12T03:24:48.790819', 'user_id': 'U003', 'title': 'Wednesday', 'activity_status': 'paused', 'device': 'tablet', 'event_time': '2026-05-12T03:25:08.790378', 'offset': 12}
{'ingested_at': '2026-05-12T03:24:48.790846', 'user_id': 'U001', 'title': 'Stranger Things', 'activity_status': 'paused', 'device': 'mobile', 'event_time': '2026-05-12T03:25:18.790378', 'offset': 13}
{'ingested_at': '2026-05-12T03:24:48.790873', 'user_id': 'U004', 'title': 'Dark', 'activity_status': 'watching', 'device': 'laptop', 'event_time': '2026-05-12T03:25:28.790378', 'offset': 14}
{'ingested_at': '20


# Retrieve Message by Offset


In [51]:

message = netflix_topic.get_messages_by_offset(19)

print(message)


{'ingested_at': '2026-05-12T03:24:48.790991', 'user_id': 'U006', 'title': 'The Crown', 'activity_status': 'watching', 'device': 'smart_tv', 'event_time': '2026-05-12T03:26:18.790378', 'offset': 19}



# Latest Activities


In [52]:

consumer.view_latest_activity()

consumer.view_latest_activity('U001')
consumer.view_latest_activity('U002')
consumer.view_latest_activity('U003')
consumer.view_latest_activity('U004')
consumer.view_latest_activity('U005')
consumer.view_latest_activity('U006')



Latest Activity:
{'ingested_at': '2026-05-12T03:24:48.790991', 'user_id': 'U006', 'title': 'The Crown', 'activity_status': 'watching', 'device': 'smart_tv', 'event_time': '2026-05-12T03:26:18.790378', 'offset': 19}

Latest Activity:
{'ingested_at': '2026-05-12T03:24:48.790967', 'user_id': 'U001', 'title': 'Stranger Things', 'activity_status': 'closed', 'device': 'mobile', 'event_time': '2026-05-12T03:26:08.790378', 'offset': 18}

Latest Activity:
{'ingested_at': '2026-05-12T03:24:48.790897', 'user_id': 'U002', 'title': 'Money Heist', 'activity_status': 'closed', 'device': 'tv', 'event_time': '2026-05-12T03:25:38.790378', 'offset': 15}

Latest Activity:
{'ingested_at': '2026-05-12T03:24:48.790945', 'user_id': 'U003', 'title': 'Wednesday', 'activity_status': 'watching', 'device': 'tablet', 'event_time': '2026-05-12T03:25:58.790378', 'offset': 17}

Latest Activity:
{'ingested_at': '2026-05-12T03:24:48.790873', 'user_id': 'U004', 'title': 'Dark', 'activity_status': 'watching', 'device': '

{'ingested_at': '2026-05-12T03:24:48.790991',
 'user_id': 'U006',
 'title': 'The Crown',
 'activity_status': 'watching',
 'device': 'smart_tv',
 'event_time': '2026-05-12T03:26:18.790378',
 'offset': 19}


# Message Count


In [53]:

print('Total Messages:', netflix_topic.get_message_count())


Total Messages: 20



# Delete Message by Offset


In [54]:

netflix_topic.delete_messages(2)

for msg in netflix_topic.view_all_messages():
    print(msg)


{'ingested_at': '2026-05-12T03:24:48.790582', 'user_id': 'U001', 'title': 'Stranger Things', 'activity_status': 'watching', 'device': 'mobile', 'event_time': '2026-05-12T03:24:48.790378', 'offset': 10}
{'ingested_at': '2026-05-12T03:24:48.790785', 'user_id': 'U002', 'title': 'Money Heist', 'activity_status': 'watching', 'device': 'tv', 'event_time': '2026-05-12T03:24:58.790378', 'offset': 11}
{'ingested_at': '2026-05-12T03:24:48.790819', 'user_id': 'U003', 'title': 'Wednesday', 'activity_status': 'paused', 'device': 'tablet', 'event_time': '2026-05-12T03:25:08.790378', 'offset': 12}
{'ingested_at': '2026-05-12T03:24:48.790846', 'user_id': 'U001', 'title': 'Stranger Things', 'activity_status': 'paused', 'device': 'mobile', 'event_time': '2026-05-12T03:25:18.790378', 'offset': 13}
{'ingested_at': '2026-05-12T03:24:48.790873', 'user_id': 'U004', 'title': 'Dark', 'activity_status': 'watching', 'device': 'laptop', 'event_time': '2026-05-12T03:25:28.790378', 'offset': 14}
{'ingested_at': '20


# Poll / Consume Messages


In [55]:

consumer.poll_messages()

consumer.poll_by_user('U001')
consumer.poll_by_user('U002')
consumer.poll_by_user('U003')
consumer.poll_by_user('U004')
consumer.poll_by_user('U005')
consumer.poll_by_user('U006')



Consumed Messages:
{'ingested_at': '2026-05-12T03:24:48.790582', 'user_id': 'U001', 'title': 'Stranger Things', 'activity_status': 'watching', 'device': 'mobile', 'event_time': '2026-05-12T03:24:48.790378', 'offset': 10}
{'ingested_at': '2026-05-12T03:24:48.790785', 'user_id': 'U002', 'title': 'Money Heist', 'activity_status': 'watching', 'device': 'tv', 'event_time': '2026-05-12T03:24:58.790378', 'offset': 11}
{'ingested_at': '2026-05-12T03:24:48.790819', 'user_id': 'U003', 'title': 'Wednesday', 'activity_status': 'paused', 'device': 'tablet', 'event_time': '2026-05-12T03:25:08.790378', 'offset': 12}
{'ingested_at': '2026-05-12T03:24:48.790846', 'user_id': 'U001', 'title': 'Stranger Things', 'activity_status': 'paused', 'device': 'mobile', 'event_time': '2026-05-12T03:25:18.790378', 'offset': 13}
{'ingested_at': '2026-05-12T03:24:48.790873', 'user_id': 'U004', 'title': 'Dark', 'activity_status': 'watching', 'device': 'laptop', 'event_time': '2026-05-12T03:25:28.790378', 'offset': 14}

[]


# Queue Behaviour Justification

This implementation uses Queue (FIFO) behaviour.

Reasons:
1. Netflix activity streams should preserve event order.
2. Older events are processed before newer events.
3. Queue-based processing resembles Kafka-like systems.
4. collections.deque efficiently supports FIFO operations.



# Reflection

## What Worked Well
- Queue-based streaming simulation
- Inheritance implementation
- Latest activity tracking

## Future Improvements
- Real-time Kafka integration
- Multiple consumers
- Distributed streaming support
